# Documentary Speaker Classifier
**Pipeline:** diarized transcript → per-speaker dossiers → rule pre-pass → Gemma 4 31B joint classification

Each cell is self-contained. Run top to bottom. Change `TRANSCRIPT_FILES` in Cell 3 to point at your files.


## 1 · Install / imports

In [2]:
import re
import json
import textwrap
from pathlib import Path
from collections import defaultdict
import os
os.environ["TRANSFORMERS_MOE_IMPLEMENTATION"] = "eager"

import torch
from transformers import AutoProcessor, AutoModelForCausalLM

print(f"torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.0f} GB")

torch 2.8.0+cu128  |  CUDA available: True
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition  102 GB


## 2 · Configuration

In [3]:
# ── Edit these paths to point at your transcript files ──────────────────────
TRANSCRIPT_FILES = {
}

MODEL_ID = "google/gemma-4-26B-A4B-it"

# Thinking OFF → faster, deterministic enough for classification
ENABLE_THINKING = False

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Lower temp than default for more consistent JSON
GENERATION_KWARGS = dict(
    max_new_tokens=20480,
    do_sample=False,
    num_assistant_tokens=4,
)


## 3 · Pipeline functions

In [4]:
# ── Parser ────────────────────────────────────────────────────────────────────
# Handles [MM:SS] or [H:MM:SS] timestamps, e.g. [03:30] or [01:00:00]
LINE_RE = re.compile(r"\[(\d+(?::\d+)+)\]\s+([^:]+):\s*(.*)$")

# Strip any trailing parenthetical/bracketed tag so "AMEENA" and "AMEENA (VO)" /
# "COBE (LAUGHING)" / "AMEENA (VO]" (typo) all collapse to one speaker
VO_TAG_RE = re.compile(r"\s*[\(\[][^)\]]*[\)\]]\s*$")

# Guard against action/description lines that accidentally end in a colon
# (e.g. "AMEENA PUSHES A GUY ... SHOUTING.:") getting misread as a speaker name
MAX_SPEAKER_NAME_LEN = 40

def normalize_speaker(name: str) -> str:
    return VO_TAG_RE.sub("", name).strip()

def parse_transcript(text: str) -> list[dict]:
    utterances = []
    skipped = 0
    for line in text.strip().splitlines():
        m = LINE_RE.match(line.strip())
        if m:
            speaker = normalize_speaker(m.group(2).strip())
            body = m.group(3).strip()
            if len(speaker) > MAX_SPEAKER_NAME_LEN or not body:
                skipped += 1
                continue
            utterances.append({
                "time":    m.group(1),
                "speaker": speaker,
                "text":    body,
            })
    if skipped:
        print(f"  (skipped {skipped} malformed/action lines)")
    return utterances

# ── Dossier builder ───────────────────────────────────────────────────────────
def _sample(utterances, n_head=3, n_tail=2):
    return [u["text"] for u in utterances]  # send all

def build_dossiers(utterances: list[dict]) -> dict[str, dict]:
    speakers = defaultdict(lambda: {"utterances": [], "word_count": 0,
                                    "question_count": 0, "total_utterances": 0})
    total_words = sum(len(u["text"].split()) for u in utterances)
    for u in utterances:
        spk = u["speaker"]
        speakers[spk]["utterances"].append({"time": u["time"], "text": u["text"]})
        speakers[spk]["word_count"]      += len(u["text"].split())
        speakers[spk]["total_utterances"] += 1
        if u["text"].strip().endswith("?"):
            speakers[spk]["question_count"] += 1

    dossiers = {}
    for spk, data in speakers.items():
        wc, uc, qc = data["word_count"], data["total_utterances"], data["question_count"]
        dossiers[spk] = {
            "speaker_id":       spk,
            "utterances":       data["utterances"],
            "word_count":       wc,
            "utterance_count":  uc,
            "question_count":   qc,
            "question_ratio":   round(qc / uc, 3) if uc else 0,
            "share_of_words":   round(wc / total_words, 3) if total_words else 0,
            "sample_utterances": _sample(data["utterances"]),
            "rule_labels":      [],
            "rule_evidence":    [],
        }
    return dossiers


# ── Rule-based pre-pass ───────────────────────────────────────────────────────
NAME_INTRO_RE   = re.compile(r"\bmy name is ([A-Z][a-z]+ ?[A-Z]?[a-z]*)", re.IGNORECASE)
SELF_ROLE_RE    = re.compile(
    r"\b(I am|I'm) (a |an )?(doctor|surgeon|physician|nurse|police|officer|"
    r"detective|chief|reporter|journalist|anchor|professor|researcher|pastor|"
    r"reverend|politician|senator|congressman|mayor|governor|activist)\b", re.IGNORECASE)
NARRATION_RE    = re.compile(
    r"\b(\d{1,2}) years ago\b|\bI (traveled|returned|decided|witnessed)\b"
    r"|\bmy (book|film|documentary)\b", re.IGNORECASE)
THIRD_PARTY_RE  = re.compile(r"\b(Miss|Ms\.|Mrs\.|Mr\.) ([A-Z][a-z]+)", re.IGNORECASE)

def apply_rules(dossiers: dict[str, dict]) -> dict[str, dict]:
    for spk, d in dossiers.items():
        all_text = " ".join(u["text"] for u in d["utterances"])

        m = NAME_INTRO_RE.search(all_text)
        if m:
            d["detected_name"] = m.group(1)
            d["rule_labels"].append("identified_by_self_intro")
            d["rule_evidence"].append(f'Self-intro: "{m.group(0)}"')

        m = SELF_ROLE_RE.search(all_text)
        if m:
            d["rule_labels"].append("professional")
            d["rule_evidence"].append(f'Role declaration: "{m.group(0)}"')

        m = NARRATION_RE.search(all_text)
        if m:
            d["rule_labels"].append("narrator_interviewer")
            d["rule_evidence"].append(f'Narration marker: "{m.group(0)}"')

        if d["question_ratio"] >= 0.35 and d["utterance_count"] >= 5:
            if "narrator_interviewer" not in d["rule_labels"]:
                d["rule_labels"].append("narrator_interviewer")
            d["rule_evidence"].append(
                f'High question ratio: {d["question_ratio"]:.0%} '
                f'({d["question_count"]}/{d["utterance_count"]} utterances)')

        for u in d["utterances"]:
            m = THIRD_PARTY_RE.search(u["text"])
            if m:
                d.setdefault("third_party_names_mentioned", []).append(m.group(2))

    return dossiers


# ── Prompt builder ────────────────────────────────────────────────────────────
LABEL_RUBRIC = """
## Label Definitions (multi-label allowed per speaker)

- **narrator_interviewer**: Sets scenes in past tense, addresses the audience directly, and/or
  asks the majority of questions. Usually only one per documentary.

- **bereaved**: Loss and grief is the PRIMARY reason they are in the documentary. They are
  interviewed because of who they lost, not because of a role or cause.

- **family_friend**: Has a personal relationship to the central subject or cause, but that
  relationship is background context rather than the point. Not primarily grieving.

- **advocate_progun**: Advocates for gun rights, Second Amendment protections, armed self-defense,
  or against gun regulation. May be organized or simply ideological — what matters is a deliberate
  pro-gun stance, not just casual gun ownership.

- **advocate_reform**: Advocates for gun control, policy change, violence prevention, or systemic
  reform. Includes bereaved parents turned activists, community organizers, and researchers who
  take an explicit policy position.

- **professional**: Speaks with institutional authority by title or role — doctor, nurse, police,
  researcher, lawyer, academic. Should self-identify or speak with clear institutional framing.

- **eyewitness**: Was present at or directly affected by a specific event, but not primarily
  grieving. Describes what they saw or experienced firsthand.

- **community_voice**: Speaks from lived experience in a place or community without professional
  authority or organized advocacy. Expresses a worldview rather than a deliberate position.

- **interviewee**: Catch-all for someone who doesn't fit a more specific category. Do NOT assign
  alongside a more specific label — if something fits, use that instead.

- **news_clip**: Fragmented broadcast audio, 911 call, or news report with no narrative context.

- **performer**: Delivers spoken word, rap, song, or other clearly scripted artistic piece.

- **unknown**: Fewer than 3 utterances AND no strong signal. Do not use if any other label fits.

## Priority rules
1. narrator_interviewer wins over all others if speaker asks most questions AND sets scenes.
2. Drop interviewee if any more specific label fits — it is a last resort.
3. unknown only when fewer than 3 utterances and nothing else is inferable.
4. advocate_progun and advocate_reform are mutually exclusive — a speaker cannot hold both.
5. Do not assign advocate_progun to someone who merely owns guns or expresses a worldview —
   that is community_voice. Advocacy requires awareness of being in a debate.

## Examples

SPEAKER_C (18 utterances, question_ratio=0.0, share=0.21):
  Sample: "My son Daniel loved basketball.",
          "The day they told me he was gone I was at work.",
          "I fight every day because of him."
→ Labels: ["bereaved", "advocate_reform"]
  Evidence: First-person grief about deceased child plus active present-tense fighting framing.

SPEAKER_G (14 utterances, question_ratio=0.0, share=0.09):
  Sample: "He wanted to be the first black president.",
          "Lewis never walked anywhere. The day he walked was the day he was killed.",
          "We fight because of the violence happening in our community."
→ Labels: ["bereaved", "family_friend", "advocate_reform"]
  Evidence: Speaks about deceased son and explicitly frames their presence around fighting
  violence — all three labels apply simultaneously.
"""

def _roster_block(dossiers: dict) -> str:
    roster_parts = []
    for spk, d in dossiers.items():
        rule_note = ""
        if d["rule_labels"]:
            rule_note = (f"\n  [Rule pre-pass: {', '.join(d['rule_labels'])}] "
                         f"Evidence: {'; '.join(d['rule_evidence'])}")
        sample_block = "\n    ".join(f'"{t}"' for t in d["sample_utterances"])
        roster_parts.append(
            f"--- {spk} ---\n"
            f"  utterances={d['utterance_count']}, words={d['word_count']}, "
            f"share={d['share_of_words']:.1%}, question_ratio={d['question_ratio']:.0%}"
            f"{rule_note}\n"
            f"  Sample utterances:\n    {sample_block}"
        )
    return "\n\n".join(roster_parts)

def build_prompt(dossiers: dict, doc_title: str) -> str:
    return (
        f'You are classifying speakers in a documentary transcript titled "{doc_title}".\n'
        f"{LABEL_RUBRIC}\n"
        f'## Speaker Roster for "{doc_title}"\n\n'
        + _roster_block(dossiers)
        + """

## Task

Classify every speaker above. Return a JSON array — one object per speaker — with these fields:
- "speaker_id": string (exactly as shown above)
- "labels": array of label strings (multi-label allowed)
- "primary_label": the single most important label
- "confidence": "high" | "medium" | "low"
- "display_name": short descriptive name if inferable, otherwise null
- "evidence": 1-2 sentences citing the utterance(s) or signals that drove classification

Return ONLY the JSON array. No preamble, no markdown fences, no trailing text.
"""
    )

def build_prompt_compact(dossiers: dict, doc_title: str) -> str:
    """Lighter-weight prompt for the minor/one-off-speaker retry pass.
    Same rubric (labeling quality shouldn't drop), but a smaller output
    schema per speaker so many more speakers fit under the token cap."""
    return (
        f'You are classifying MINOR/background speakers in a documentary transcript '
        f'titled "{doc_title}". Each has very few lines, so keep each answer brief.\n'
        f"{LABEL_RUBRIC}\n"
        f'## Speaker Roster (minor speakers) for "{doc_title}"\n\n'
        + _roster_block(dossiers)
        + """

## Task

Classify every speaker above. Return a JSON array — one object per speaker — with ONLY these
fields (no evidence, no display_name):
- "speaker_id": string — the speaker's bare name only, exactly as it appears between the "---"
  markers in the roster above (e.g. for a roster entry "--- DISPATCHER 1 ---", the speaker_id
  is "DISPATCHER 1", WITHOUT the surrounding dashes)
- "labels": array of label strings (multi-label allowed)
- "primary_label": the single most important label
- "confidence": "high" | "medium" | "low"

Return ONLY the JSON array. No preamble, no markdown fences, no trailing text, no explanations.
"""
    )

def split_dossiers(dossiers: dict, min_utterances: int = 2, min_words: int = 15):
    """Split into major (gets full prompt) and minor (gets compact prompt) speakers."""
    major, minor = {}, {}
    for spk, d in dossiers.items():
        if d["utterance_count"] >= min_utterances or d["word_count"] >= min_words:
            major[spk] = d
        else:
            minor[spk] = d
    return major, minor

print("Pipeline functions loaded ✓")

Pipeline functions loaded ✓


## 4 · Load Gemma 4 31B

In [5]:
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
import torch

MODEL_ID = "google/gemma-4-26B-A4B-it"
ASSISTANT_ID = "google/gemma-4-26B-A4B-it-assistant"

print("Loading model ...")
print("(first run downloads ~60 GB — subsequent runs use the HF cache)\n")

processor = AutoProcessor.from_pretrained(MODEL_ID)

# 1. Main model config — force eager MoE implementation
config = AutoConfig.from_pretrained(MODEL_ID)
config._experts_implementation = "eager"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)

# 2. Assistant model config — force eager MoE implementation
assistant_config = AutoConfig.from_pretrained(ASSISTANT_ID)
assistant_config._experts_implementation = "eager"

assistant_model = AutoModelForCausalLM.from_pretrained(
    ASSISTANT_ID,
    config=assistant_config,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)

model.eval()
assistant_model.eval()

print("\nModels loaded ✓")
print(f"Dtype : {next(model.parameters()).dtype}")
print(f"Device: {next(model.parameters()).device}")
print(f"Assistant dtype : {next(assistant_model.parameters()).dtype}")
print(f"Assistant device: {next(assistant_model.parameters()).device}")

Loading model ...
(first run downloads ~60 GB — subsequent runs use the HF cache)



Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/48 [00:00<?, ?it/s]


Models loaded ✓
Dtype : torch.bfloat16
Device: cuda:0
Assistant dtype : torch.bfloat16
Assistant device: cuda:0


## 5 · Inference helper

In [6]:
def run_model(prompt: str) -> tuple[str, bool]:
    import time

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt},
    ]
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

    t0 = time.time()
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    t1 = time.time()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            assistant_model=assistant_model,
            **GENERATION_KWARGS,
        )
        t2 = time.time()

    output_tokens = outputs.shape[-1] - input_len
    truncated = output_tokens >= GENERATION_KWARGS["max_new_tokens"]
    print(f"  Input tokens: {input_len:,} | Output tokens: {output_tokens:,}"
          f"{'  ⚠ TRUNCATED' if truncated else ''}")
    print(f"  Prefill: {t1-t0:.1f}s | Generation: {t2-t1:.1f}s | Total: {t2-t0:.1f}s")

    decoded = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    parsed = processor.parse_response(decoded)

    del inputs, outputs
    torch.cuda.empty_cache()

    if isinstance(parsed, dict):
        text_out = parsed.get("text", "") or parsed.get("content", "") or str(parsed)
    else:
        text_out = parsed
    return text_out, truncated


def parse_json_response(raw) -> list[dict]:
    if isinstance(raw, dict):
        raw = raw.get("text", "") or raw.get("content", "") or json.dumps(raw)
    raw = raw.strip()
    raw = re.sub(r"^```[a-z]*\n?", "", raw)
    raw = re.sub(r"\n?```$",       "", raw)
    bracket_end = raw.rfind("]")
    if bracket_end != -1:
        raw = raw[:bracket_end + 1]
    return json.loads(raw)

print("Inference helpers ready ✓")

Inference helpers ready ✓


## 6 · Run the full pipeline

In [7]:
# all_results = {}   # doc_title → list of classification dicts

# for doc_title, filepath in TRANSCRIPT_FILES.items():
#     path = Path(filepath)
#     if not path.exists():
#         print(f"⚠  File not found: {filepath}  — skipping")
#         continue

#     print(f"\n{'='*60}")
#     print(f"Processing: {doc_title}")
#     print(f"{'='*60}")

#     # ── Parse + dossiers + rules ──────────────────────────────
#     text       = path.read_text(encoding="utf-8")
#     utterances = parse_transcript(text)
#     dossiers   = build_dossiers(utterances)
#     dossiers   = apply_rules(dossiers)

#     print(f"  Speakers   : {len(dossiers)}")
#     print(f"  Utterances : {len(utterances)}")

#     rule_hits = [(s, d["rule_labels"]) for s, d in dossiers.items() if d["rule_labels"]]
#     if rule_hits:
#         print("  Rule pre-pass hits:")
#         for spk, labels in rule_hits:
#             print(f"    {spk}: {labels}")

#     # ── Attempt 1: single pass over everyone ──────────────────
#     print("\n  Calling model ...", flush=True)
#     prompt = build_prompt(dossiers, doc_title)
#     raw_resp, truncated = run_model(prompt)

#     results = None
#     if not truncated:
#         try:
#             results = parse_json_response(raw_resp)
#         except (json.JSONDecodeError, AttributeError, TypeError) as e:
#             print(f"  ⚠  JSON parse error on full pass: {e}")
#             truncated = True  # treat as truncation-shaped failure, try the split path

#     # ── Fallback: only if the full pass hit the token cap / failed to parse ──
#     if truncated:
#         print("  ⚠  Full pass truncated or unparseable — splitting into major/minor passes")
#         major, minor = split_dossiers(dossiers)
#         print(f"  Major speakers: {len(major)}  |  Minor speakers: {len(minor)}")

#         print("\n  Calling model on major speakers ...", flush=True)
#         prompt_major = build_prompt(major, doc_title)
#         raw_major, trunc_major = run_model(prompt_major)
#         try:
#             results = parse_json_response(raw_major)
#         except (json.JSONDecodeError, AttributeError, TypeError) as e:
#             print(f"  ⚠  JSON parse error on major pass: {e}")
#             print("  Raw response (first 500 chars):", raw_major[:500])
#             continue
#         if trunc_major:
#             print("  ⚠  STILL truncated on major-only pass — consider raising "
#                   "min_utterances/min_words in split_dossiers()")

#         if minor:
#             print("\n  Calling model on minor speakers (compact schema) ...", flush=True)
#             prompt_minor = build_prompt_compact(minor, doc_title)
#             raw_minor, trunc_minor = run_model(prompt_minor)
#             try:
#                 minor_results = parse_json_response(raw_minor)
#                 results += minor_results
#             except (json.JSONDecodeError, AttributeError, TypeError) as e:
#                 print(f"  ⚠  JSON parse error on minor pass: {e}")
#                 print("  Raw response (first 500 chars):", raw_minor[:500])
#                 print(f"  ({len(minor)} minor speakers left unclassified for this doc)")

#     if results is None:
#         continue

#     # ── Merge classifications back into dossiers ───────────────
#     result_map = {r["speaker_id"]: r for r in results}
#     for spk, d in dossiers.items():
#         d["classification"] = result_map.get(spk)

#     all_results[doc_title] = {
#         "dossiers":        dossiers,
#         "classifications":  results,
#     }
#     print(f"\n  Done — {len(results)} speakers classified ✓")

## 7 · Display results

In [8]:
# LABEL_ICONS = {
#     "narrator_interviewer": "🎬",
#     "subject":              "⭐",
#     "family_friend":        "💛",
#     "professional":         "🏥",
#     "activist_advocate":    "✊",
#     "news_clip":            "📡",
#     "performer":            "🎤",
#     "unknown":              "❓",
# }

# CONF_ICON = {"high": "🟢", "medium": "🟡", "low": "🔴"}

# for doc_title, data in all_results.items():
#     print(f"\n{'═'*65}")
#     print(f"  {doc_title}")
#     print(f"{'═'*65}")

#     results = sorted(
#         data["classifications"],
#         key=lambda r: data["dossiers"].get(r["speaker_id"], {}).get("word_count", 0),
#         reverse=True,
#     )

#     for r in results:
#         spk  = r["speaker_id"]
#         d    = data["dossiers"].get(spk, {})
#         icon = CONF_ICON.get(r.get("confidence", "low"), "⚪")
#         labels_str = "  ".join(
#             f'{LABEL_ICONS.get(l, "·")} {l}' for l in r.get("labels", [])
#         )
#         name = r.get("display_name") or ""
#         print(f"\n  {spk:<14}  {icon} {r.get('confidence','?'):6}  {labels_str}")
#         if name:
#             print(f"  {'':14}  → {name}")
#         print(f"  {'':14}  utterances={d.get('utterance_count','?')}  "
#               f"words={d.get('word_count','?')}  "
#               f"q_ratio={d.get('question_ratio',0):.0%}")
#         evidence = r.get("evidence", "")
#         if evidence:
#             wrapped = textwrap.fill(evidence, width=60,
#                                     initial_indent="  "*3,
#                                     subsequent_indent="  "*3)
#             print(wrapped)


## 8 · Run it on the folder

In [10]:
import os
import json
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_FOLDER  = "transcripts/missing"
OUTPUT_FOLDER = "transcripts_labeled"
OVERWRITE_EXISTING = False

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

# ── Label → short token (for building speaker IDs) ───────────────────────────
LABEL_ORDER = [
    "narrator_interviewer", "bereaved", "family_friend", "advocate_progun",
    "advocate_reform", "professional", "eyewitness", "community_voice",
    "interviewee", "news_clip", "performer", "unknown",
]

LABEL_TOKEN = {
    "narrator_interviewer": "NARRATOR_INTERVIEWER",
    "bereaved":             "BEREAVED",
    "family_friend":        "FAMILY_FRIEND",
    "advocate_progun":      "ADVOCATE_PROGUN",
    "advocate_reform":      "ADVOCATE_REFORM",
    "professional":         "PROFESSIONAL",
    "eyewitness":           "EYEWITNESS",
    "community_voice":      "COMMUNITY_VOICE",
    "interviewee":          "INTERVIEWEE",
    "news_clip":            "NEWS_CLIP",
    "performer":            "PERFORMER",
    "unknown":              "UNKNOWN",
}


def build_speaker_id_map(classifications: list[dict]) -> dict[str, str]:
    combo_counters = {}
    speaker_map    = {}
    for r in classifications:
        # Strip stray "---" wrapping in case the model echoes roster formatting
        spk = re.sub(r"^-+\s*|\s*-+$", "", r["speaker_id"]).strip()
        labels = r.get("labels", ["unknown"])
        sorted_labels = sorted(labels, key=lambda l: LABEL_ORDER.index(l)
                               if l in LABEL_ORDER else 999)
        token = "_".join(LABEL_TOKEN.get(l, l.upper()) for l in sorted_labels)
        combo_counters[token] = combo_counters.get(token, 0) + 1
        n = combo_counters[token]
        speaker_map[spk] = f"{token}_{n:02d}"
    return speaker_map


def relabel_transcript(text: str, speaker_map: dict[str, str]) -> str:
    output_lines = []
    for line in text.splitlines():
        m = LINE_RE.match(line.strip())
        if m:
            time = m.group(1)
            spk  = m.group(2).strip()
            utt  = m.group(3).strip()
            new_spk = speaker_map.get(spk, spk)
            output_lines.append(f"[{time}] {new_spk}: {utt}")
        else:
            output_lines.append(line)
    return "\n".join(output_lines)


def is_valid_transcript(text: str) -> bool:
    matches = sum(1 for line in text.splitlines() if LINE_RE.match(line.strip()))
    return matches >= 5


# ── Main batch loop ───────────────────────────────────────────────────────────
txt_files = sorted(Path(INPUT_FOLDER).glob("*.txt"))
print(f"Found {len(txt_files)} .txt files in '{INPUT_FOLDER}'\n")

for txt_path in txt_files:
    out_path = Path(OUTPUT_FOLDER) / txt_path.name

    if not OVERWRITE_EXISTING and out_path.exists():
        print(f"⏩ Skipping: {txt_path.name} (Already exists in output folder)")
        continue

    print(f"{'='*60}")
    print(f"Processing: {txt_path.name}")

    text = txt_path.read_text(encoding="utf-8")

    if not is_valid_transcript(text):
        print(f"  ⚠  Skipping — does not match expected transcript format")
        continue

    # ── Parse + dossiers + rules ──────────────────────────────
    utterances = parse_transcript(text)
    dossiers   = build_dossiers(utterances)
    dossiers   = apply_rules(dossiers)

    print(f"  Speakers: {len(dossiers)}  |  Utterances: {len(utterances)}")

    rule_hits = [(s, d["rule_labels"]) for s, d in dossiers.items() if d["rule_labels"]]
    if rule_hits:
        print("  Rule pre-pass hits:")
        for spk, labels in rule_hits:
            print(f"    {spk}: {labels}")

    # ── Classify: attempt 1, full roster in one pass ──────────
    print("  Calling model ...", flush=True)
    prompt = build_prompt(dossiers, txt_path.stem)
    raw_resp, truncated = run_model(prompt)

    classifications = None
    if not truncated:
        try:
            classifications = parse_json_response(raw_resp)
        except (json.JSONDecodeError, AttributeError, TypeError) as e:
            print(f"  ⚠  Parse error ({type(e).__name__}): {e}")
            truncated = True  # fall through to split path

    # ── Fallback: only if the full pass hit the token cap / failed to parse ──
    if truncated:
        print("  ⚠  Full pass truncated or unparseable — splitting into major/minor passes")
        major, minor = split_dossiers(dossiers)
        print(f"  Major speakers: {len(major)}  |  Minor speakers: {len(minor)}")

        prompt_major = build_prompt(major, txt_path.stem)
        raw_major, trunc_major = run_model(prompt_major)
        try:
            classifications = parse_json_response(raw_major)
        except (json.JSONDecodeError, AttributeError, TypeError) as e:
            print(f"  ⚠  Parse error on major pass ({type(e).__name__}): {e}")
            print(f"  Raw response (first 300 chars): {str(raw_major)[:300]}")
            continue
        if trunc_major:
            print("  ⚠  STILL truncated on major-only pass — consider raising "
                  "min_utterances/min_words in split_dossiers()")

        if minor:
            prompt_minor = build_prompt_compact(minor, txt_path.stem)
            raw_minor, _ = run_model(prompt_minor)
            try:
                classifications += parse_json_response(raw_minor)
            except (json.JSONDecodeError, AttributeError, TypeError) as e:
                print(f"  ⚠  Parse error on minor pass ({type(e).__name__}): {e}")
                print(f"  ({len(minor)} minor speakers left unclassified for this file)")

    if classifications is None:
        continue

    # ── Build speaker map + relabel ───────────────────────────
    first_appearance = {spk: i for i, spk in enumerate(dossiers.keys())}
    classifications_sorted = sorted(
        classifications,
        key=lambda r: first_appearance.get(r["speaker_id"], 999)
    )

    speaker_map = build_speaker_id_map(classifications_sorted)

    unmatched = [spk for spk in dossiers if spk not in speaker_map]
    if unmatched:
        print(f"  ⚠  {len(unmatched)} speakers in transcript have NO classification match "
              f"(will be left unrelabeled): {unmatched}")

    print("  Speaker mapping:")
    for orig, new in speaker_map.items():
        print(f"    {orig:<16} → {new}")

    relabeled_text = relabel_transcript(text, speaker_map)

    if relabeled_text == text:
        print("  ⚠  WARNING: relabeled output is IDENTICAL to input — "
              "speaker_map likely didn't match any transcript speaker IDs. Not saving.")
        continue

    print("  Speaker mapping:")
    for orig, new in speaker_map.items():
        print(f"    {orig:<16} → {new}")

    relabeled_text = relabel_transcript(text, speaker_map)

    out_path.write_text(relabeled_text, encoding="utf-8")
    print(f"  Saved → {out_path}")

print(f"\n{'='*60}")
print(f"Done. Relabeled files written to '{OUTPUT_FOLDER}/'")

Found 5 .txt files in 'transcripts/missing'

⏩ Skipping: Charm City.txt (Already exists in output folder)
Processing: Gunned Down.txt
  Speakers: 35  |  Utterances: 443
  Calling model ...
  Input tokens: 14,057 | Output tokens: 3,033
  Prefill: 0.1s | Generation: 118.3s | Total: 118.3s
  ⚠  1 speakers in transcript have NO classification match (will be left unrelabeled): ["Ed O'Keefe"]
  Speaker mapping:
    UNKNOWN          → NEWS_CLIP_01
    Will Lyman       → NARRATOR_INTERVIEWER_01
    Barack Obama     → PROFESSIONAL_01
    Joe Biden        → PROFESSIONAL_02
    John Aquilino    → ADVOCATE_PROGUN_PROFESSIONAL_01
    Charlton Heston  → ADVOCATE_PROGUN_01
    Pat Maisch       → EYEWITNESS_01
    Mark Kelly       → FAMILY_FRIEND_01
    Dennis Henigan   → INTERVIEWEE_01
    Ed O'KEEFE       → PROFESSIONAL_03
    Paul Barrett     → PROFESSIONAL_04
    Tom Mauser       → BEREAVED_ADVOCATE_REFORM_01
    Dylan Klebold    → INTERVIEWEE_02
    Richard Feldman  → PROFESSIONAL_05
    J. Warre